# Portfolio Verdicts via MiniMax Function Calling

SDK request plus a separate `curl` version of the same payload.

In [ ]:
%pip install openai

In [ ]:
import json
import os

from openai import OpenAI

API_KEY = os.getenv("OPENAI_API_KEY", "")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.minimax.io/v1")
MODEL = os.getenv("OPENAI_MODEL", "MiniMax-M2.7-highspeed")

assert API_KEY, "Set OPENAI_API_KEY in your environment"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [ ]:
request_payload = {
    "selected_advisors": [
        {"id": "warren_buffett", "name": "Warren Buffett", "role": "Long-term value investor"},
        {"id": "pavel_durov", "name": "Pavel Durov", "role": "Tech growth and product focus"},
        {"id": "tyler_durden", "name": "Tyler Durden", "role": "High-conviction contrarian"}
    ],
    "risk_profile": "medium",
    "cash_balance_usdt": "1000.00",
    "portfolio": {
        "total_balance_usdt": "1000.00",
        "pnl_percent": "0.00",
        "pnl_absolute": "0.00",
        "assets": []
    },
    "market_snapshots": [
        {"asset_id": "HOODx", "price": "75.13", "upside_percent": "12.40"},
        {"asset_id": "GOOGLx", "price": "314.20", "upside_percent": "8.10"},
        {"asset_id": "NVDAx", "price": "921.50", "upside_percent": "6.25"}
    ]
}

messages = [
    {
        "role": "user",
        "content": (
            "Using the selected advisors, portfolio, cash balance, risk profile, and market snapshots, "
            "decide a verdict for each asset. Return the result through the function call only. Input: "
            + json.dumps(request_payload, ensure_ascii=True)
        ),
    }
]

tools = [
    {
        "type": "function",
        "function": {
            "name": "emit_portfolio_verdicts",
            "description": "Return portfolio verdicts for the provided assets.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "verdicts": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "asset_id": {"type": "string"},
                                "verdict": {"type": "string", "enum": ["buy", "hold", "sell"]},
                                "reason": {"type": "string"}
                            },
                            "required": ["asset_id", "verdict", "reason"],
                            "additionalProperties": false
                        }
                    }
                },
                "required": ["summary", "verdicts"],
                "additionalProperties": false
            }
        }
    }
]

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "emit_portfolio_verdicts"}},
    temperature=0,
    max_tokens=220,
    timeout=5,
    extra_body={"reasoning_split": True},
)

message = response.choices[0].message
message

In [40]:
# Same request via curl
curl_payload = {
    "model": MODEL,
    "messages": messages,
    "tools": tools,
    "tool_choice": {"type": "function", "function": {"name": "emit_portfolio_verdicts"}},
    "temperature": 0,
    "max_tokens": 220,
    "extra_body": {"reasoning_split": True}
}

curl_command = (
    f"curl -s {BASE_URL.rstrip('/')}/chat/completions "
    f"-H 'Content-Type: application/json' "
    f"-H 'Authorization: Bearer {API_KEY}' "
    f"-d '{json.dumps(curl_payload, ensure_ascii=True)}'"
)

print(curl_command)

curl -s https://api.minimax.io/v1/chat/completions -H 'Content-Type: application/json' -H 'Authorization: Bearer <OPENAI_API_KEY_FROM_ENV>' -d '{"model": "MiniMax-M2.7-highspeed", "messages": [{"role": "user", "content": "Using the selected advisors, portfolio, cash balance, risk profile, and market snapshots, decide a verdict for each asset. Return the result through the function call only. Input: {\"selected_advisors\": [{\"id\": \"warren_buffett\", \"name\": \"Warren Buffett\", \"role\": \"Long-term value investor\"}, {\"id\": \"pavel_durov\", \"name\": \"Pavel Durov\", \"role\": \"Tech growth and product focus\"}, {\"id\": \"tyler_durden\", \"name\": \"Tyler Durden\", \"role\": \"High-conviction contrarian\"}], \"risk_profile\": \"medium\", \"cash_balance_usdt\": \"1000.00\", \"portfolio\": {\"total_balance_usdt\": \"1000.00\", \"pnl_percent\": \"0.00\", \"pnl_absolute\": \"0.00\", \"assets\": []}, \"market_snapshots\": [{\"asset_id\": \"HOODx\", \"price\": \"75.13\", \"upside_perc

In [39]:
message.reasoning_details

[{'type': 'reasoning.text',
  'id': 'reasoning-text-1',
  'format': 'MiniMax-response-v1',
  'index': 0,
  'text': 'Let me analyze this request carefully.\n\nI need to provide investment verdicts for each asset in the market snapshots based on:\n1. Selected advisors: Warren Buffett (value investor), Pavel Durov (tech growth), Tyler Durden (contrarian)\n2. Risk profile: medium\n3. Cash balance: $1000\n4. Portfolio: Empty (all cash)\n5. Market snapshots with assets: HOODx, GOOGLx, NVDAx\n\nLet me think about each asset:\n\n**HOODx (Price: $75.13, Upside: 12.40%)**\n- HOOD is Robinhood Markets, a fintech company\n- 12.40% upside potential is moderate-high\n- Tech-focused asset, relevant for Durov\n- Moderate upside fits medium risk profile\n- Reasonable valuation candidate\n\n**GOOGLx (Price: $314.20, Upside: 8.10%)**\n- GOOGL is Google/Alphabet\n- 8.10% upside is more conservative\n- Blue-chip tech with stable'}]

In [ ]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    print(json.dumps(arguments, indent=2, ensure_ascii=False))
else:
    print(message.content)